# Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Check data integration between all sources

In [0]:
TABLE_CONFIG = [
  "bronze.crm_cust_info",
  "bronze.crm_prd_info",
  "bronze.crm_sales_details",
  "bronze.erp_cust_az12",
  "bronze.erp_loc_a101",
  "bronze.erp_px_cat_g1v2"
]

for table in TABLE_CONFIG:
  spark.read.table(table).limit(50).display()

# Read the table

In [0]:
df = spark.read.table("bronze.crm_sales_details")

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Check is there any duplicate data or null values in sls_ord_num

In [0]:
df.groupBy("sls_ord_num").agg(count("sls_ord_num")).\
    filter((col("sls_ord_num").isNull()) | (count("sls_ord_num") > 1)).display()

In [0]:
df.select("*").filter(col("sls_ord_num").isin("SO51176", "SO51177", "SO51178")).display()

so, we have a tons of duplicates data, but, it seems like because if one customer ordering mupltiple product, it will be recorded in the same ord_num, so it's nice, we wont gonna do anything about it. in other hand, we dont have any null values, which is good

# Check is there any null values in sls_prd_key and sls_cust_id

In [0]:
df.select("*").filter((col("sls_prd_key").isNull()) | (col("sls_cust_id").isNull())).display()

# Check the null data in date data

In [0]:
df.select("*").filter((col("sls_order_dt").isNull()) | (col("sls_ship_dt").isNull()) | (col("sls_due_dt").isNull())).display()

# Check the correctness of date data

In [0]:
df.select("*").filter((col("sls_order_dt") > col("sls_ship_dt")) | 
    (col("sls_order_dt") > col("sls_due_dt")) |
    (col("sls_ship_dt") > col("sls_due_dt"))
    ).display()

# Clean the date data

In [0]:
df = (df.withColumn("sls_order_dt",
               when((col("sls_order_dt").isNull()) | (length(col("sls_order_dt")) != 8), None)
               .otherwise(to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")))
      .withColumn("sls_ship_dt",
          when((col("sls_ship_dt").isNull()) | (length(col("sls_ship_dt")) != 8), None)
          .otherwise(to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
      )
      .withColumn("sls_due_dt",
          when((col("sls_due_dt").isNull()) | (length(col("sls_due_dt")) != 8), None)
          .otherwise(to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")))
     )

# Check the correctness of sls_sales, sls_quantity, and sls_price

In [0]:
df.select("*").filter(col("sls_price") * col("sls_quantity") != col("sls_sales")).display()

# Transform the sls_price, sls_quantity, sls_sales:

Since we have a lot of uncorrect data, lets clean them  with rules:
- if the sales negative, 0, null, or unmatch with price*quantity, then transform based on price and quantitiy value
- if the price zero and null, match the value based on sales and quantitiy
- if price is negative, transform it into positive


In [0]:
df.withColumn(
    "sls_sales",
    when(col("sls_sales") <= 0, abs(col("sls_quantity") * col("sls_price")))
    .when(col("sls_sales").isNull(), abs(col("sls_quantity") * col("sls_price")))
    .when(col("sls_sales") != col("sls_quantity") * col("sls_price"), abs(col("sls_quantity") * col("sls_price"))).otherwise(col("sls_sales"))
).withColumn(
    "sls_price",
    when(col("sls_price") <= 0, abs(col("sls_sales") / col("sls_quantity")))
    .when(col("sls_price").isNull(), abs(col("sls_sales") / col("sls_quantity")))
    .when(col("sls_price") != col("sls_sales") / col("sls_quantity"), abs(col("sls_sales") / col("sls_quantity"))).otherwise(col("sls_price"))
).filter((col("sls_price") * col("sls_quantity") != col("sls_sales")) |
         (col("sls_price").isNull()) | (col("sls_sales").isNull())).display()

In [0]:
df = df.withColumn(
    "sls_sales",
    when(col("sls_sales") <= 0, abs(col("sls_quantity") * col("sls_price")))
    .when(col("sls_sales").isNull(), abs(col("sls_quantity") * col("sls_price")))
    .when(col("sls_sales") != col("sls_quantity") * col("sls_price"), abs(col("sls_quantity") * col("sls_price"))).otherwise(col("sls_sales"))
).withColumn(
    "sls_price",
    when(col("sls_price") <= 0, abs(col("sls_sales") / col("sls_quantity")))
    .when(col("sls_price").isNull(), abs(col("sls_sales") / col("sls_quantity")))
    .when(col("sls_price") != col("sls_sales") / col("sls_quantity"), abs(col("sls_sales") / col("sls_quantity"))).otherwise(col("sls_price"))
)

# Sanity Check

In [0]:
df.display()